**ANN - Regression**

In [87]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [88]:
# Reading data
data = pd.read_csv('C:/MAFAS/APU/CT046-3-M-AML/CT046 - LABS/Python LAB MATERIALS/Lab 9 - Neural Network/mart.csv')
data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [89]:
data = data.drop(['Item_Identifier', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Location_Type'], axis = 1)

In [90]:
data.dtypes

Item_Weight          float64
Item_Fat_Content      object
Item_Visibility      float64
Item_Type             object
Item_MRP             float64
Outlet_Size           object
Outlet_Type           object
Item_Outlet_Sales    float64
dtype: object

In [91]:
data.isnull().sum()

Item_Weight          1463
Item_Fat_Content        0
Item_Visibility         0
Item_Type               0
Item_MRP                0
Outlet_Size          2410
Outlet_Type             0
Item_Outlet_Sales       0
dtype: int64

In [92]:
# Assign variables
x = data.drop('Item_Outlet_Sales', axis = 1)
y = data['Item_Outlet_Sales']

In [93]:
# Data Split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [94]:
# Impute missing values for Item_Weight
Item_Weight_imputer = SimpleImputer(strategy = 'mean')
x_train['Item_Weight'] = Item_Weight_imputer.fit_transform(x_train[['Item_Weight']])
x_test['Item_Weight'] = Item_Weight_imputer.transform(x_test[['Item_Weight']])

In [95]:
# Impute missing values for Outlet_Size
Outlet_Size_imputer = SimpleImputer(strategy = 'most_frequent')
x_train['Outlet_Size'] = Outlet_Size_imputer.fit_transform(x_train[['Outlet_Size']]).ravel()
x_test['Outlet_Size'] = Outlet_Size_imputer.transform(x_test[['Outlet_Size']]).ravel()

In [96]:
x_train.isnull().sum()

Item_Weight         0
Item_Fat_Content    0
Item_Visibility     0
Item_Type           0
Item_MRP            0
Outlet_Size         0
Outlet_Type         0
dtype: int64

In [97]:
# Label Encoding for categorical variables
Item_Fat_Content_le = LabelEncoder()
Item_Fat_Content_le.fit(x_train['Item_Fat_Content'])
x_train['Item_Fat_Content'] = Item_Fat_Content_le.transform(x_train['Item_Fat_Content'])
x_test['Item_Fat_Content'] = Item_Fat_Content_le.transform(x_test['Item_Fat_Content'])

Item_Type_le = LabelEncoder()
x_train['Item_Type'] = Item_Type_le.fit_transform(x_train['Item_Type'])
x_test['Item_Type'] = Item_Type_le.transform(x_test['Item_Type'])

Outlet_Size_le = LabelEncoder()
x_train['Outlet_Size'] = Outlet_Size_le.fit_transform(x_train['Outlet_Size'])
x_test['Outlet_Size'] = Outlet_Size_le.transform(x_test['Outlet_Size'])

Outlet_Type_le = LabelEncoder()
x_train['Outlet_Type'] = Outlet_Type_le.fit_transform(x_train['Outlet_Type'])
x_test['Outlet_Type'] = Outlet_Type_le.transform(x_test['Outlet_Type'])

In [98]:
# Standardization
scaler = StandardScaler()
x_train = pd.DataFrame(scaler.fit_transform(x_train), columns = x_train.columns)
x_test = pd.DataFrame(scaler.transform(x_test), columns = x_test.columns)

In [99]:
x_train

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Size,Outlet_Type
0,-0.801383,0.973699,-0.600703,-0.302126,0.470709,-0.284198,-0.259489
1,1.210152,-0.571618,-0.362159,0.411711,0.457877,-0.284198,-0.259489
2,1.115491,0.973699,0.194933,0.649656,-0.482625,1.383325,-0.259489
3,-1.079448,-0.571618,-0.704944,-0.302126,-1.603553,-0.284198,-0.259489
4,-0.008602,-0.571618,1.383177,1.363492,0.218375,1.383325,-0.259489
...,...,...,...,...,...,...,...
6813,-0.826231,0.973699,4.282848,-0.302126,-0.043511,-0.284198,-1.509802
6814,0.642189,-0.571618,1.001006,-0.540071,-1.059078,-0.284198,-0.259489
6815,1.115491,-0.571618,-0.916931,0.173765,1.526207,-0.284198,-0.259489
6816,1.766282,2.519016,-0.228187,1.363492,-0.383072,-0.284198,-0.259489


**ANN Regressor - Base Model**

In [100]:
model = Sequential()
model.add(Input(shape = (7,))) # IL
model.add(Dense(units = 256, kernel_initializer = 'he_normal', activation = 'relu')) # HL1
model.add(Dropout(0.2))
model.add(Dense(units = 128, kernel_initializer = 'he_normal', activation = 'relu')) # HL2
model.add(Dropout(0.2))
model.add(Dense(units = 1, kernel_initializer = 'glorot_uniform', activation = 'linear')) # OL

model.compile(loss='mean_squared_error', optimizer = 'adam', metrics = ['mse'])

model.fit(x_train, y_train, validation_split = 0.1, batch_size = 16, epochs = 100)

Epoch 1/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 4493907.5000 - mse: 4493907.5000 - val_loss: 2124349.7500 - val_mse: 2124349.7500
Epoch 2/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1745988.1250 - mse: 1745988.1250 - val_loss: 1847251.8750 - val_mse: 1847251.8750
Epoch 3/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1627623.6250 - mse: 1627623.6250 - val_loss: 1768781.7500 - val_mse: 1768781.7500
Epoch 4/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1570603.5000 - mse: 1570603.5000 - val_loss: 1702350.2500 - val_mse: 1702350.2500
Epoch 5/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1516368.1250 - mse: 1516368.1250 - val_loss: 1636777.2500 - val_mse: 1636777.2500
Epoch 6/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1472755.8750 - mse: 1472755.8750 - val_loss: 1582271.0000 - val_mse: 1582271.0000
Epoch 7/100
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1437349.3750 - mse: 1437349.3750 - val_loss: 1551828.5000 - val_mse: 1551828.5000

In [101]:
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
print('MSE:', round(mean_squared_error(y_pred, y_test),2))
print('RMSE:', round(root_mean_squared_error(y_pred, y_test),2))
print('R2:', round(r2_score(y_pred, y_test),2))

54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step
MSE: 1045669.39
RMSE: 1022.58
R2: 0.35


**Using Keras Tuner**

In [102]:
# %pip install keras-tuner

In [103]:
import keras_tuner as kt

def build_model(hp):
    model = Sequential()
    model.add(Input(shape = (7,))) # IL

    hp_units_1 = hp.Int('HL1', min_value = 16, max_value = 256, step = 4)
    hp_dropout_1 = hp.Float('Dropout1', min_value = 0.0, max_value = 0.5, step = 0.1)
    hp_units_2 = hp.Int('HL2', min_value = 16, max_value = 128, step = 4)
    hp_dropout_2 = hp.Float('Dropout2', min_value = 0.0, max_value = 0.5, step = 0.1)
    
    model.add(Dense(units = hp_units_1, activation = 'relu'))
    model.add(Dropout(hp_dropout_1))
    model.add(Dense(units = hp_units_2, activation = 'relu'))
    model.add(Dropout(hp_dropout_2))
    model.add(Dense(1, activation = 'linear'))
    
    hp_optimizer = hp.Choice('Optimizer', ['sgd', 'adam', 'rmsprop'])
    
    model.compile(optimizer = hp_optimizer, loss = 'mean_squared_error', metrics = ['mse'])
    return model

tuner = kt.RandomSearch(build_model, objective = 'val_mse', max_trials = 5, overwrite = True)

tuner.search(x_train, y_train, validation_split = 0.1, epochs = 100, batch_size = 16)

Trial 5 Complete [00h 00m 39s]
val_mse: nan

Best val_mse So Far: 1355054.25
Total elapsed time: 00h 03m 24s


In [104]:
for h_param in ["HL1", "Dropout1", "HL2", "Dropout2", "Optimizer"]:
    print(h_param, tuner.get_best_hyperparameters()[0].get(h_param))


HL1 196
Dropout1 0.1
HL2 48
Dropout2 0.4
Optimizer adam


In [105]:
tuner.results_summary(num_trials = 5)

Results summary
Results in .\untitled_project
Showing 5 best trials
Objective(name="val_mse", direction="min")

Trial 1 summary
Hyperparameters:
HL1: 196
Dropout1: 0.1
HL2: 48
Dropout2: 0.4
Optimizer: adam
Score: 1355054.25

Trial 3 summary
Hyperparameters:
HL1: 56
Dropout1: 0.1
HL2: 20
Dropout2: 0.1
Optimizer: adam
Score: 1399663.0

Trial 0 summary
Hyperparameters:
HL1: 56
Dropout1: 0.30000000000000004
HL2: 20
Dropout2: 0.2
Optimizer: adam
Score: 1566065.0

Trial 2 summary
Hyperparameters:
HL1: 248
Dropout1: 0.2
HL2: 100
Dropout2: 0.30000000000000004
Optimizer: sgd
Score: nan

Trial 4 summary
Hyperparameters:
HL1: 160
Dropout1: 0.30000000000000004
HL2: 80
Dropout2: 0.4
Optimizer: sgd
Score: nan


**Final Model**

In [106]:
best_model = tuner.get_best_models(num_models = 1)[0]

c:\Users\raheem\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [107]:
y_pred_f = best_model.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
print('MSE:', round(mean_squared_error(y_pred_f, y_test),2))
print('RMSE:', round(root_mean_squared_error(y_pred_f, y_test),2))
print('R2:', round(r2_score(y_pred_f, y_test),2))

54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
MSE: 1067130.76
RMSE: 1033.02
R2: 0.32
